In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error


df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')


test_ids = df_test['id']


df_train = df_train.drop(columns=['id', 'int_col'], errors='ignore')
df_test = df_test.drop(columns=['id', 'int_col'], errors='ignore')


def clean_transmission(value):
    if pd.isna(value): return 'Other'
    val = str(value).lower()
    if 'cvt' in val: return 'CVT'
    elif 'manual' in val or 'm/t' in val: return 'Manual'
    elif 'auto-shift' in val or 'dct' in val or 'dual shift' in val: return 'Dual-Clutch'
    elif 'a/t' in val or 'automatic' in val or 'overdrive' in val: return 'Automatic'
    elif 'speed' in val and ('mt' in val or 'm/t' in val): return 'Manual'
    else: return 'Other'


df_train['transmission'] = df_train['transmission'].apply(clean_transmission)
df_test['transmission'] = df_test['transmission'].apply(clean_transmission)

def extract_engine_features(df_input):
    df = df_input.copy() 
    df['horsepower'] = df['engine'].str.extract(r'(\d+\.?\d*)(?=HP)').astype(float)
    df['engine_size'] = df['engine'].str.extract(r'(\d+\.?\d*)(?=L)').astype(float)
    df['cylinders'] = df['engine'].str.extract(r'(\d+)\s(?:Cylinder|V\d|Straight)')[0].astype(float)
    df = df.drop('engine', axis=1) 
    return df


df_train = extract_engine_features(df_train)
df_test = extract_engine_features(df_test)


mean_horsepower_train = df_train['horsepower'].mean()
mean_engine_size_train = df_train['engine_size'].mean()
mean_cylinders_train = df_train['cylinders'].mean()


df_train['horsepower'] = df_train['horsepower'].fillna(mean_horsepower_train).round()
df_train['engine_size'] = df_train['engine_size'].fillna(mean_engine_size_train).round()
df_train['cylinders'] = df_train['cylinders'].fillna(mean_cylinders_train).round()

df_test['horsepower'] = df_test['horsepower'].fillna(mean_horsepower_train).round()
df_test['engine_size'] = df_test['engine_size'].fillna(mean_engine_size_train).round()
df_test['cylinders'] = df_test['cylinders'].fillna(mean_cylinders_train).round()



df_train['fuel_type'] = df_train['fuel_type'].replace(['–', 'not supported'], np.nan)
df_test['fuel_type'] = df_test['fuel_type'].replace(['–', 'not supported'], np.nan)


moda_fuel_type = df_train['fuel_type'].mode()[0]
df_train['fuel_type'] = df_train['fuel_type'].fillna(moda_fuel_type)
df_test['fuel_type'] = df_test['fuel_type'].fillna(moda_fuel_type) 


df_train['accident'] = df_train['accident'].fillna('None reported')
df_test['accident'] = df_test['accident'].fillna('None reported')

df_train['clean_title'] = df_train['clean_title'].fillna('No')
df_test['clean_title'] = df_test['clean_title'].fillna('No')



X = df_train.drop('price', axis=1)
y = df_train['price']


categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=np.number).columns.tolist() 


num_train_rows = len(X)
combined_data = pd.concat([X, df_test], ignore_index=True)


le_model = LabelEncoder()
combined_data['model'] = le_model.fit_transform(combined_data['model'])

le_brand = LabelEncoder()
combined_data['brand'] = le_brand.fit_transform(combined_data['brand'])


ohe_cols = [col for col in categorical_cols if col not in ['model', 'brand']]

encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
encoded_features = encoder.fit_transform(combined_data[ohe_cols])
encoded_feature_names = encoder.get_feature_names_out(ohe_cols)
encoded_df = pd.DataFrame(encoded_features, columns=encoded_feature_names, index=combined_data.index)


combined_data_processed = pd.concat([combined_data.drop(columns=ohe_cols), encoded_df], axis=1)


X_processed = combined_data_processed.iloc[:num_train_rows]
df_test_processed = combined_data_processed.iloc[num_train_rows:]


print("\n--- Debugging: Tipi di dati in X_processed prima della scalatura ---")
print(X_processed.dtypes)
print("\n--- Debugging: Valori NaN totali in X_processed ---")
print(X_processed.isnull().sum().sum())

scaler = StandardScaler()


all_numerical_cols_for_scaling = [col for col in X_processed.columns if X_processed[col].dtype in ['int64', 'float64']]

X_processed[all_numerical_cols_for_scaling] = scaler.fit_transform(X_processed[all_numerical_cols_for_scaling])
df_test_processed[all_numerical_cols_for_scaling] = scaler.transform(df_test_processed[all_numerical_cols_for_scaling])


X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)


print("\nInizio addestramento modello SVR iniziale...")
svr_model = SVR(kernel='rbf', C=100, epsilon=0.1)
svr_model.fit(X_train, y_train) 


y_pred_val = svr_model.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f"RMSE del modello SVR (su set di validazione): {rmse_val:.4f}")


print("\nInizio Grid Search per ottimizzazione degli iperparametri...")
param_grid = {
    'C': [1, 10, 100],
    'epsilon': [0.05, 0.1, 0.2],
    'kernel': ['rbf'],
    'gamma': ['scale', 0.01, 0.1]
}

grid_search = GridSearchCV(SVR(), param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

print(f"\nMigliori parametri: {grid_search.best_params_}")
best_svr_model = grid_search.best_estimator_

y_pred_best_val = best_svr_model.predict(X_val)
rmse_best_val = np.sqrt(mean_squared_error(y_val, y_pred_best_val))
print(f"RMSE del modello SVR (migliore da Grid Search su set di validazione): {rmse_best_val:.4f}")


print("\nAddestramento del modello finale su tutto il set di training processato...")
best_svr_model.fit(X_processed, y)


final_test_predictions = best_svr_model.predict(df_test_processed)


final_test_predictions[final_test_predictions < 0] = 0


submission_df = pd.DataFrame({'id': test_ids, 'price': final_test_predictions})
submission_df.to_csv('submission.csv', index=False)
print("\nFile 'submission.csv' creato con successo!")


--- Debugging: Tipi di dati in X_processed prima della scalatura ---
brand                                       int64
model                                       int64
model_year                                  int64
milage                                      int64
horsepower                                float64
                                           ...   
ext_col_designo Diamond White Bright      float64
ext_col_designo Diamond White Metallic    float64
ext_col_–                                 float64
accident_None reported                    float64
clean_title_Yes                           float64
Length: 335, dtype: object

--- Debugging: Valori NaN totali in X_processed ---
0


C:\Users\Roman\AppData\Local\Temp\ipykernel_18972\4251114233.py:134: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_processed[all_numerical_cols_for_scaling] = scaler.fit_transform(X_processed[all_numerical_cols_for_scaling])
C:\Users\Roman\AppData\Local\Temp\ipykernel_18972\4251114233.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_processed[all_numerical_cols_for_scaling] = scaler.transform(df_test_processed[all_numerical_cols_for_scaling])



Inizio addestramento modello SVR iniziale...
